# Lesson 3: Agentic Search

In [1]:
# libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [2]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]


'The Blackwell GPU architecture features a second-generation Transformer Engine, advanced NVLink, and is designed for generative AI. It includes a 10 TB/s interconnect and is built on a 4NP process.'

## Regular search

In [3]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

> Note: search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [4]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

returning previous results due to exception reaching ddg.
https://weather.com/weather/today/l/USCA0987:1:US
https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8


In [5]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


> Note: This produces a long output, you may want to right click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [7]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:5000]) # limit long outputs

returning previous results due to exception reaching ddg.
Website: https://weather.com/weather/today/l/USCA0987:1:US


<body><div class="appWrapper DaybreakLargeScreen LargeScreen lightTheme twcTheme DaybreakLargeScreen--appWrapper--ZkDop gradients--sunnyDay--MqMid gradients--sunnyDay-top--Bldhb" id="appWrapper"><div class="region-meta"><div class="removeIfEmpty" id="WxuHtmlHead-meta-"></div><div class="removeIfEmpty" id="WxuNewsroom-meta-bc9f40d5-d941-4fd8-bae2-2d8d63a38bb3"></div></div><div class="regionHeaderWrapper DaybreakLargeScreen--stickyHeader--0Pgu8"><div class="regionHeaderInnerWrapper"><div class="adsSectionOuterWrapper"><div class="adsSectionInnerWrapper"><div class="stickyAdPlaceholder"></div><div class="js-branded-background-ads" id="labBG"></div><div class="js-branded-background-ads" id="wx-hero-content"></div><div class="region-stickyAds regionStickyAds"><div class="removeIfEmpty" id="WxuAd-stickyAds-50b69813-b340-4d89-a22d-016d4b682491"><div class="adWrapper BaseAd--a

In [8]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/weather/today/l/USCA0987:1:US


Recent Locations Menu Weather Forecasts Radar & Maps News & Media Products & Account Lifestyle Specialty Forecasts San Francisco, CA Small Craft Advisory Weather Today in San Francisco, CA 6:15 am 8:16 pm Hourly Weather - San Francisco, CA Now Sunny 7 pm Sunny 8 pm Sunny 9 pm Clear Don't Miss Seasonal Hub 10 Day Weather - San Francisco, CA Tonight Night Clear skies. Low around 55F. Winds W at 15 to 25 mph. Mon 04 Day Sunny. Becoming windy late. High 69F. Winds W at 20 to 30 mph. Night Partly cloudy. Low 54F. Winds W at 15 to 25 mph. Tue 05 Day Mostly cloudy early, then afternoon sunshine. High 67F. Winds WSW at 10 to 20 mph. Night Mostly clear. Low near 55F. Winds SW at 10 to 15 mph. Wed 06 Day A few clouds from time to time. High 73F. Winds WSW at 10 to 20 mph. Night Generally fair. Low 57F. Winds W at 10 to 15 mph. Radar Travel We Love Our Critters Summer And Your Skin Home, Garage & Garden August Outlook Keeping You Health

## Agentic Search

In [9]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1754265637, 'localtime': '2025-08-03 17:00'}, 'current': {'last_updated_epoch': 1754265600, 'last_updated': '2025-08-03 17:00', 'temp_c': 17.0, 'temp_f': 62.6, 'is_day': 1, 'condition': {'text': 'Mist', 'icon': '//cdn.weatherapi.com/weather/64x64/day/143.png', 'code': 1030}, 'wind_mph': 15.2, 'wind_kph': 24.5, 'wind_degree': 270, 'wind_dir': 'W', 'pressure_mb': 1019.0, 'pressure_in': 30.09, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 83, 'cloud': 50, 'feelslike_c': 17.0, 'feelslike_f': 62.6, 'windchill_c': 14.1, 'windchill_f': 57.3, 'heatindex_c': 14.8, 'heatindex_f': 58.7, 'dewpoint_c': 12.5, 'dewpoint_f': 54.4, 'vis_km': 9.7, 'vis_miles': 6.0, 'uv': 3.0, 'gust_mph': 20.6, 'gust_kph': 33.1}}


In [10]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1754265637,
        "localtime": "2025-08-03 17:00"
    },
    "current": {
        "last_updated_epoch": 1754265600,
        "last_updated": "2025-08-03 17:00",
        "temp_c": 17.0,
        "temp_f": 62.6,
        "is_day": 1,
        "condition": {
            "text": "Mist",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/143.png",
            "code": 1030
        },
        "wind_mph": 15.2,
        "wind_kph": 24.5,
        "wind_degree": 270,
        "wind_dir": "W",
        "pressure_mb": 1019.0,
        "pressure_in": 30.09,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 83,
        "cloud": 50,
        "feelslike_c": 17.0,
        "feelslike_f": 62.6,
        "windchill_c": 14.1,
        "windc

<img src="./google_sample.png" width="800" height="600">